In [ ]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier, Pool
import torch


# =========================
# Config
# =========================
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    SAVE_OOF = True
    SAVE_PRED = True
    SAVE_SUB = True

    MODEL_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.0,
        "grow_policy": "SymmetricTree",
        "bootstrap_type": "Bayesian",
        "random_seed": SEED,
        "verbose": 200,
        "early_stopping_rounds": 200,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
    }


# =========================
# Column groups
# =========================
NUM_COLS_BASE = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

CAT_COLS_BASE = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

BIN_COLS_BASE = [
    "Mulching_Used",
]

CAT_ALL_BASE = CAT_COLS_BASE + BIN_COLS_BASE
FEATURE_COLS_BASE = NUM_COLS_BASE + CAT_COLS_BASE + BIN_COLS_BASE


# =========================
# Utilities
# =========================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def load_data():
    train_df = pd.read_csv(CFG.TRAIN_PATH)
    test_df = pd.read_csv(CFG.TEST_PATH)
    return train_df, test_df


def validate_columns(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    expected_train = set(FEATURE_COLS_BASE + [CFG.TARGET_COL, CFG.ID_COL])
    expected_test = set(FEATURE_COLS_BASE + [CFG.ID_COL])

    train_missing = expected_train - set(train_df.columns)
    train_extra = set(train_df.columns) - expected_train

    test_missing = expected_test - set(test_df.columns)
    test_extra = set(test_df.columns) - expected_test

    print("train_missing:", train_missing)
    print("train_extra:", train_extra)
    print("test_missing:", test_missing)
    print("test_extra:", test_extra)

    if train_missing or test_missing:
        raise ValueError("Column mismatch detected.")


def cast_dtypes(train_df: pd.DataFrame, test_df: pd.DataFrame):
    # CatBoostに素直に渡すため、カテゴリ系は string に寄せる
    for col in CAT_ALL_BASE:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)

    # 数値は数値のまま
    for col in NUM_COLS_BASE:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    return train_df, test_df


def make_pools(X_tr, y_tr, X_va, y_va, X_te):
    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_te,
        cat_features=CAT_ALL_BASE,
    )
    return train_pool, valid_pool, test_pool


def apply_class_bias(proba: np.ndarray, bias: np.ndarray | None = None) -> np.ndarray:
    # baselineでは bias なし。後でOOF上最適化する余地を残すだけ。
    if bias is None:
        return proba
    return proba + bias.reshape(1, -1)


# =========================
# Main CV
# =========================
def main():
    seed_everything(CFG.SEED)

    train_df, test_df = load_data()
    validate_columns(train_df, test_df)
    train_df, test_df = cast_dtypes(train_df, test_df)

    X = train_df[FEATURE_COLS_BASE].copy()
    X_test = test_df[FEATURE_COLS_BASE].copy()
    y_raw = train_df[CFG.TARGET_COL].copy()

    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    class_names = list(le.classes_)
    n_classes = len(class_names)

    print("classes:", class_names)
    print("train shape:", train_df.shape)
    print("test shape:", test_df.shape)

    skf = StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    )

    oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
    test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        print("=" * 60)
        print(f"fold {fold} / {CFG.N_SPLITS}")
        print("=" * 60)

        X_tr = X.iloc[tr_idx].copy()
        X_va = X.iloc[va_idx].copy()
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        train_pool, valid_pool, test_pool = make_pools(X_tr, y_tr, X_va, y_va, X_test)

        model = CatBoostClassifier(**CFG.MODEL_PARAMS)

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True,
        )

        va_proba = model.predict_proba(valid_pool)
        te_proba = model.predict_proba(test_pool)

        oof_proba[va_idx] = va_proba
        test_proba += te_proba / CFG.N_SPLITS

        va_pred = np.argmax(va_proba, axis=1)
        fold_bacc = balanced_accuracy_score(y_va, va_pred)
        fold_scores.append(fold_bacc)

        print(f"fold {fold} balanced_accuracy = {fold_bacc:.6f}")

    print("=" * 60)
    print("fold scores:", [round(s, 6) for s in fold_scores])
    print(f"cv mean balanced_accuracy = {np.mean(fold_scores):.6f}")
    print(f"cv std  balanced_accuracy = {np.std(fold_scores):.6f}")

    # OOF全体でのBalanced Accuracy
    oof_pred = np.argmax(oof_proba, axis=1)
    oof_bacc = balanced_accuracy_score(y, oof_pred)
    print(f"OOF balanced_accuracy = {oof_bacc:.6f}")

    # クラス別 recall 確認
    print("\nPer-class recall on OOF:")
    for cls_idx, cls_name in enumerate(class_names):
        mask = (y == cls_idx)
        recall = (oof_pred[mask] == y[mask]).mean()
        print(f"{cls_name}: {recall:.6f}")

    # submission
    test_pred = np.argmax(test_proba, axis=1)
    test_label = le.inverse_transform(test_pred)

    sub_df = pd.DataFrame({
        CFG.ID_COL: test_df[CFG.ID_COL],
        CFG.TARGET_COL: test_label
    })

    if CFG.SAVE_OOF:
        np.save("oof_catboost_strict_baseline_proba.npy", oof_proba)

    if CFG.SAVE_PRED:
        np.save("pred_catboost_strict_baseline_proba.npy", test_proba)

    if CFG.SAVE_SUB:
        sub_df.to_csv("submission_catboost_strict_baseline.csv", index=False)

    print("\nSaved:")
    if CFG.SAVE_OOF:
        print("- oof_catboost_strict_baseline_proba.npy")
    if CFG.SAVE_PRED:
        print("- pred_catboost_strict_baseline_proba.npy")
    if CFG.SAVE_SUB:
        print("- submission_catboost_strict_baseline.csv")


if __name__ == "__main__":
    main()